# Reconocimiento de placas en via de lastre

Ejecuta las celdas **en orden**. Hay una sola pausa obligatoria, en la celda 3,
que pide reiniciar el entorno.

**Antes de empezar:** activa la GPU en `Entorno de ejecucion > Cambiar tipo de
entorno de ejecucion > GPU`. Cambiarla despues reinicia todo.

## 1. Ver que GPU y que CUDA hay

In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
!nvcc --version | grep release
import sys; print("Python", sys.version.split()[0])

## 2. Montar tu Google Drive

Aprueba la ventana emergente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Instalar dependencias

`onnxruntime-gpu` se fija a la version 1.22.0 a proposito. Las versiones mas
nuevas estan compiladas contra CUDA 13, y Colab trae CUDA 12: cargan mal y el
proceso cae a procesador sin avisar.

La instalacion **no** va en modo silencioso, para que puedas ver que version
quedo si algo falla.

**Al terminar, reinicia el entorno** (`Entorno de ejecucion > Reiniciar entorno
de ejecucion`) y sigue en la celda 4.

In [ ]:
!pip install -q opencv-python-headless numpy openpyxl pillow
!pip install -q fast-alpr open-image-models

# La version importa: 1.23 en adelante exige CUDA 13, que Colab no tiene.
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install "onnxruntime-gpu==1.22.0"

print()
print("=" * 62)
print("REINICIA EL ENTORNO DE EJECUCION Y SIGUE EN LA CELDA 4")
print("=" * 62)

## 4. Traer el codigo

Se puede ejecutar las veces que quieras.

In [ ]:
import os

PROYECTO = "/content/lastre"

if os.path.isdir(f"{PROYECTO}/.git"):
    !cd $PROYECTO && git pull --quiet
else:
    !rm -rf $PROYECTO
    !git clone --quiet https://github.com/riofutabac/PLACAS-RECONOCIMIENTO.git $PROYECTO

%cd $PROYECTO
!git log --oneline -1

## 5. Comprobar si la GPU quedo activa

Que `CUDAExecutionProvider` aparezca en la lista de compilados no significa
nada: es lo que el paquete trae, no lo que funciona. Esta celda crea una
sesion real y, si falla, muestra el error exacto de CUDA.

In [ ]:
import onnxruntime as ort
from open_image_models import create_detector

print("onnxruntime:", ort.__version__)
print("compilados :", ort.get_available_providers())
print()

detector = create_detector("rf-detr-nano-384-coco",
                           providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
activos = detector.model.get_providers()
print("activos    :", activos)
print()

GPU_OK = any("CUDA" in p or "Tensorrt" in p for p in activos)

if GPU_OK:
    print("GPU ACTIVA. Sigue en la celda 6.")
else:
    print("GPU NO ACTIVA. El detalle del fallo:")
    print()
    try:
        ort.InferenceSession(
            "/root/.cache/open-image-models/rf-detr-nano-384-coco.onnx",
            providers=["CUDAExecutionProvider"])
    except Exception as err:
        print(str(err)[:800])
    print()
    !ldconfig -p | grep -E "libcublasLt|libcudnn" | head -5
    print()
    print("Si dice que falta libcublasLt.so.13 o pide CUDA 13, la version de")
    print("onnxruntime-gpu es demasiado nueva. Prueba en la celda 3 con 1.21.1")
    print("o 1.20.2. Si nada funciona, usa --acelerador cpu en las celdas 8 y 10:")
    print("el proceso corre igual, solo que mas lento.")

## 6. Indicar donde estan tus videos

Cambia la ruta si tu carpeta se llama distinto. Si te compartieron la carpeta,
abrela en Drive y usa **Organizar > Anadir acceso directo** hacia Mi unidad.

In [ ]:
from pathlib import Path

CARPETA_VIDEOS = "/content/drive/MyDrive/Cam PL"
CARPETA_SALIDA = "/content/drive/MyDrive/informe_lastre"

# Cambia a "cpu" si la celda 5 dijo que la GPU no esta activa
ACELERADOR = "gpu"

videos = sorted(Path(CARPETA_VIDEOS).glob("*.mp4"))
print(f"Videos encontrados: {len(videos)}")
for v in videos[:5]:
    print("  ", v.name)
if len(videos) > 5:
    print(f"   ... y {len(videos) - 5} mas")

assert videos, f"No hay videos .mp4 en '{CARPETA_VIDEOS}'. Revisa la ruta."
PRIMER_VIDEO = str(videos[0])

## 7. Verificar la zona de analisis

El poligono esta calibrado para esta camara en su posicion actual. Confirma
que el verde cubre la via de lastre y deja fuera la carretera principal.

Si no coincide, edita `config/zona.json` y vuelve a ejecutar esta celda.

In [ ]:
from IPython.display import Image, display

# El script elige un cuadro que exista en este video.
# Leer desde Drive es lento: deja que termine, puede tardar un par de minutos.
!cd $PROYECTO && python scripts/verificar_zona.py "$PRIMER_VIDEO"

display(Image(f"{PROYECTO}/out/verificacion_zona.jpg", width=950))

## 8. Prueba con 3 videos

Mide cuanto tarda antes de comprometer horas. La barra va de 0 a 100 sobre el
total de cuadros e incluye el tiempo restante estimado.

In [ ]:
!cd $PROYECTO && python scripts/procesar_lote.py "$CARPETA_VIDEOS" \
    --out-dir "$CARPETA_SALIDA" --acelerador $ACELERADOR --limite 3

## 9. Revisar lo que produjo la prueba

In [ ]:
from openpyxl import load_workbook

libro = load_workbook(f"{CARPETA_SALIDA}/placas_lastre.xlsx")
hoja = libro["Placas"]
print(f"Vehiculos registrados: {hoja.max_row - 1}")
print()

for fila in hoja.iter_rows(min_row=1, max_row=min(11, hoja.max_row), values_only=True):
    print("  ".join(str(x)[:16].ljust(16) for x in fila[:7]))

print()
print("--- Resumen ---")
resumen = libro["Resumen"]
for i in range(3, 20):
    clave = resumen.cell(row=i, column=1).value
    if clave:
        print(f"  {clave}: {resumen.cell(row=i, column=2).value}")

## 10. Procesar todos los videos

Agrega `--solo-salidas` si unicamente te interesan los que salen por el lastre.

**Si Colab se desconecta no pierdes nada.** El avance se guarda al terminar
cada video: vuelve a ejecutar esta misma celda y continua donde quedo.

In [ ]:
!cd $PROYECTO && python scripts/procesar_lote.py "$CARPETA_VIDEOS" \
    --out-dir "$CARPETA_SALIDA" --acelerador $ACELERADOR

## 11. Utilidades

Ejecuta solo la que necesites.

In [ ]:
# Ver cuantos videos van procesados
import json
with open(f"{CARPETA_SALIDA}/avance.json") as f:
    avance = json.load(f)
print(f"Videos procesados: {len(avance['videos'])}")
for nombre, datos in sorted(avance["videos"].items()):
    print(f"  {nombre}: {len(datos['filas'])} vehiculos")

In [ ]:
# Regenerar el Excel con lo ya procesado, sin analizar mas video
!cd $PROYECTO && python scripts/procesar_lote.py "$CARPETA_VIDEOS" \
    --out-dir "$CARPETA_SALIDA" --solo-informe

In [ ]:
# Empezar de cero, descartando el avance guardado
!cd $PROYECTO && python scripts/procesar_lote.py "$CARPETA_VIDEOS" \
    --out-dir "$CARPETA_SALIDA" --acelerador $ACELERADOR --reiniciar

## Que obtienes

En la carpeta de salida de tu Drive:

- **`placas_lastre.xlsx`**, una fila por vehiculo con su foto al lado. Verde es
  validado, amarillo pendiente de revision, rojo sin placa legible. La segunda
  hoja resume los totales.
- **`recortes/`** con las imagenes sueltas.
- **`avance.json`** con el estado del procesamiento.

Revisa a mano las filas amarillas, y tambien las verdes con consenso bajo: esa
columna dice que tan disputada fue la lectura entre cuadros.